# Phase 2 Setup: Kaggle and Colab

This notebook prepares the frozen five-language XNLI protocol, validates runtime access, defines CPU-side stream/replay/metrics/logging components, and dry-runs orchestration. **It does not load model weights or start training.**

Before running: make the repository available as a Kaggle input dataset or in Google Drive, add `HF_TOKEN` and optionally `WANDB_API_KEY` as notebook secrets, and accept the Llama model license. The Phase 2 protocol schedules 41 jobs: B0 and B2-B8 across five orders, plus one order-independent B1 joint-training job. B7/B8 source feasibility and several baseline training adapters remain flagged in the registry.

Data preparation is disabled by default; setting `PREPARE_DATA = True` downloads and writes fixed XNLI JSONL subsets but still does not train.

In [1]:
# 1. Import Required Libraries and Set Up Environment
import hashlib
import json
import os
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from peft import LoraConfig, TaskType
from transformers import AutoConfig, AutoTokenizer

try:
    import wandb
except ImportError:
    wandb = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print({"python": sys.version.split()[0], "torch": torch.__version__, "cuda_available": torch.cuda.is_available(), "wandb_available": wandb is not None})

d:\continual learning project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1110 01:36:38.244000 3452 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


{'python': '3.13.14', 'torch': '2.14.1+cu130', 'cuda_available': True, 'wandb_available': True}


In [2]:
# 2. Define Language Stream and Dataset Preparation
IS_KAGGLE = Path("/kaggle").exists()
IS_COLAB = Path("/content").exists() and not IS_KAGGLE
if IS_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

root_candidates = [Path.cwd(), *Path.cwd().parents]
if IS_KAGGLE:
    root_candidates.insert(0, Path("/kaggle/input/continual-learning-project"))
elif IS_COLAB:
    root_candidates.insert(0, Path("/content/drive/MyDrive/continual-learning-project"))
PROJECT_ROOT = next((path.resolve() for path in root_candidates if (path / "phase2/configs/pilot_protocol.json").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Mount the project repository and set PROJECT_ROOT to its root directory.")
sys.path.insert(0, str(PROJECT_ROOT))

# Read secrets from the platform secret store. Never display their values.
def _runtime_secret(name):
    try:
        if IS_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        if IS_COLAB:
            from google.colab import userdata
            return userdata.get(name)
    except Exception:
        return None
    return None

for secret_name, env_name in (("HF_TOKEN", "HF_TOKEN"), ("WANDB_API_KEY", "WANDB_API_KEY")):
    secret_value = _runtime_secret(secret_name)
    if secret_value:
        os.environ[env_name] = secret_value

from phase2.protocol import build_manifest, load_protocol
from phase2.prepare_data import prepare_xnli

PROTOCOL_PATH = PROJECT_ROOT / "phase2/configs/pilot_protocol.json"
PROTOCOL = load_protocol(PROTOCOL_PATH)
PROTOCOL_MANIFEST = build_manifest(PROTOCOL)
LANGUAGE_STREAM = PROTOCOL["dataset"]["languages"]
print({"protocol_id": PROTOCOL["protocol_id"], "languages": LANGUAGE_STREAM, "orders": len(PROTOCOL["orders"]), "scheduled_runs": PROTOCOL_MANIFEST["expected_run_count"], "buffer_capacity": PROTOCOL["replay"]["buffer_capacity_examples"]})

# Set True only when you intend to download XNLI and create the shared artifact.
PREPARE_DATA = False
PERSISTENT_OUTPUT = Path("/kaggle/working/phase2_artifacts") if IS_KAGGLE else (Path("/content/drive/MyDrive/phase2_artifacts") if IS_COLAB else PROJECT_ROOT / "phase2_artifacts")
if PREPARE_DATA:
    PREPARED_MANIFEST = prepare_xnli(PROTOCOL_PATH, PERSISTENT_OUTPUT)
    print({"manifest": PREPARED_MANIFEST["manifest_path"], "artifact_languages": len(PREPARED_MANIFEST["data_artifacts"])})

{'protocol_id': 'xnli-pilot-5lang-500x200-v1', 'languages': ['en', 'hi', 'es', 'de', 'sw'], 'orders': 5, 'scheduled_runs': 41, 'buffer_capacity': 125}


In [3]:
# 3. Implement Stream Loader
from phase2.components import StreamLoader, StreamTask


def load_prepared_split(language, split, artifact_root=PERSISTENT_OUTPUT):
    """Load one protocol-prepared JSONL split into row dictionaries."""
    path = Path(artifact_root) / "data" / f"{language}.{split}.jsonl"
    if not path.exists():
        raise FileNotFoundError(f"Prepared split not found: {path}. Run data preparation first.")
    with path.open(encoding="utf-8") as source:
        return [json.loads(line)["example"] for line in source if line.strip()]


def make_stream_loader(dataset_map, stream_spec, batch_size=1, shuffle=False, seed=SEED):
    """stream_spec is an ordered list of (language, task, split) tuples."""
    stream_tasks = [StreamTask(*item) for item in stream_spec]
    return StreamLoader(dataset_map, stream_tasks, batch_size=batch_size, shuffle=shuffle, seed=seed)

print("StreamLoader ready; shuffle is deterministic per stage when enabled.")

StreamLoader ready; shuffle is deterministic per stage when enabled.


In [4]:
# 4. Implement Replay Buffer Abstractions with Pluggable Selection Strategies
from phase2.components import (
    EmbeddingSimilaritySelector,
    GradientCoresetSelector,
    OurPriorSelector,
    UniformSelector,
)
from shared.continual_learning_lora import ReplayBuffer

REPLAY_SELECTORS = {
    "uniform": UniformSelector,
    "embedding_similarity": EmbeddingSimilaritySelector,
    "gradient_coreset": GradientCoresetSelector,
    "our_prior": OurPriorSelector,
}
REPLAY_BUFFER_CAPACITY = PROTOCOL["replay"]["buffer_capacity_examples"]

# B4's CPU selector consumes precomputed gradients. It is a gradient-matching
# adaptation; exact GCR paper parity must be confirmed before claiming reproduction.
# Embedding/gradient extraction and model-dependent replay training are deferred.
print({"buffer_capacity": REPLAY_BUFFER_CAPACITY, "selectors": sorted(REPLAY_SELECTORS)})

{'buffer_capacity': 125, 'selectors': ['embedding_similarity', 'gradient_coreset', 'our_prior', 'uniform']}


In [5]:
# 5. Implement Metrics Module (ACC, BWT, FWT, F, etc.)
from shared.continual_learning_metrics import calculate_metrics


def summarize_performance(performance_matrix, zero_shot_accuracy, task_order):
    metrics = calculate_metrics(performance_matrix, zero_shot_accuracy)
    per_task = pd.DataFrame({
        "language": task_order,
        "first_seen_accuracy": [performance_matrix[index][index] for index in range(len(task_order))],
        "final_accuracy": performance_matrix[-1],
    })
    return metrics, per_task

# Logging hooks are configured in Section 7; test splits are evaluation-only.
print("Metric functions ready:", ["ACC", "AA", "BWT", "FWT", "F", "FP"])

Metric functions ready: ['ACC', 'AA', 'BWT', 'FWT', 'F', 'FP']


In [6]:
# 6. Set Up LoRA Fine-tuning Wrapper (Skeleton, No Training)
from shared.continual_learning_lora import LlamaLoRATrainer

LORA_CONFIG = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=PROTOCOL["lora"]["rank"],
    lora_alpha=PROTOCOL["lora"]["alpha"],
    lora_dropout=PROTOCOL["lora"]["dropout"],
    target_modules=PROTOCOL["lora"]["target_modules"],
    bias="none",
)
TRAINER_SKELETON = LlamaLoRATrainer(
    model_name=PROTOCOL["model"]["id"],
    output_dir=PERSISTENT_OUTPUT / "checkpoints",
    task_mode="causal_lm",
    max_length=PROTOCOL["training"]["max_length"],
    learning_rate=PROTOCOL["training"]["learning_rate"],
    train_batch_size=PROTOCOL["training"]["train_batch_size"],
    gradient_accumulation_steps=PROTOCOL["training"]["gradient_accumulation_steps"],
    num_train_epochs=PROTOCOL["training"]["epochs"],
    seed=PROTOCOL["training"]["seeds"][0],
    lora_rank=PROTOCOL["lora"]["rank"],
    lora_alpha=PROTOCOL["lora"]["alpha"],
    lora_dropout=PROTOCOL["lora"]["dropout"],
    target_modules=PROTOCOL["lora"]["target_modules"],
    load_in_4bit=PROTOCOL["training"]["load_in_4bit"],
    gradient_checkpointing=PROTOCOL["training"]["gradient_checkpointing"],
    token=True,
)
# Intentionally do not call load_model_and_tokenizer() or train_stage() here.
print({"model": TRAINER_SKELETON.model_name, "lora_rank": LORA_CONFIG.r, "weights_loaded": TRAINER_SKELETON.model is not None})

{'model': 'meta-llama/Llama-3.2-1B', 'lora_rank': 16, 'weights_loaded': False}


In [7]:
# 7. Configure Logging System (Weights & Biases)
ENABLE_WANDB = False  # Set True only after configuring WANDB_API_KEY as a notebook secret.
WANDB_PROJECT = "asymmetric-interference-continual-learning"
TRACKING_RUN = None
LOCAL_LOG_RECORDS = []

if ENABLE_WANDB:
    if wandb is None:
        raise ImportError("Install wandb from phase2/requirements-phase2.txt first.")
    TRACKING_RUN = wandb.init(
        project=WANDB_PROJECT,
        name=f"phase2-setup-{PROTOCOL['protocol_id']}",
        config={"protocol_id": PROTOCOL["protocol_id"], "protocol_sha256": PROTOCOL_MANIFEST["protocol_sha256"]},
        mode=os.environ.get("WANDB_MODE", "online"),
    )


def log_language_accuracy(method_id, order_id, stage, language, accuracy):
    record = {"method": method_id, "order": order_id, "stage": stage, "language": language, "accuracy": float(accuracy)}
    LOCAL_LOG_RECORDS.append(record)
    if TRACKING_RUN is not None:
        TRACKING_RUN.log({f"{method_id}/{order_id}/stage_{stage}/{language}/accuracy": float(accuracy)})
    return record

print({"wandb_enabled": ENABLE_WANDB, "tracking_hook_ready": True})

{'wandb_enabled': False, 'tracking_hook_ready': True}


In [ ]:
# 8. Baseline Configuration and Experiment Orchestration
from phase2.orchestration import assignment_summary, partition_manifest_runs

BASELINE_CONFIGS = {
    method["id"]: {
        **method,
        "model": PROTOCOL["model"],
        "lora": PROTOCOL["lora"],
        "training": PROTOCOL["training"],
        "replay": PROTOCOL["replay"],
    }
    for method in PROTOCOL["methods"]
}
RUNTIME_JOBS = partition_manifest_runs(PROTOCOL_MANIFEST)
RUNTIME_SUMMARY = assignment_summary(RUNTIME_JOBS)
TRAINING_ENABLED = False
assert RUNTIME_SUMMARY["total_runs"] == PROTOCOL_MANIFEST["expected_run_count"]
print(pd.DataFrame([
    {"runtime": runtime, "jobs": info["run_count"], "methods": ", ".join(info["methods"])}
    for runtime, info in RUNTIME_SUMMARY.items() if runtime != "total_runs"
]))
print("Training enabled:", TRAINING_ENABLED)

In [10]:
# 9. Dry Run: Simulate Data Flow and Logging Without Training
DUMMY_DATASETS = {
    (language, "xnli", "train"): [
        {"id": f"{language}-{index}", "premise": f"premise {index}", "hypothesis": f"hypothesis {index}", "label": index % 3}
        for index in range(3)
    ]
    for language in LANGUAGE_STREAM
}
DUMMY_STREAM = [(language, "xnli", "train") for language in PROTOCOL["orders"][0]["languages"]]
dummy_loader = make_stream_loader(DUMMY_DATASETS, DUMMY_STREAM, batch_size=2, shuffle=True, seed=SEED)
dummy_batches = list(dummy_loader)

# Exercise the common buffer and each selector using small CPU vectors.
dummy_buffer = ReplayBuffer(capacity=3, seed=SEED)
for example in DUMMY_DATASETS[("en", "xnli", "train")]:
    dummy_buffer.add("en", example)
buffer_sample = dummy_buffer.sample(2)
selector_items = ["en-item", "hi-item", "sw-item"]
uniform_sample = UniformSelector.select(selector_items, 2, seed=SEED)
embedding_sample = EmbeddingSimilaritySelector.select(selector_items, [[1, 0], [0, 1], [-1, 0]], [1, 0], 1)
gradient_sample = GradientCoresetSelector.select(selector_items, [[-1.0], [0.0], [1.0]], 1)
prior_sample = OurPriorSelector.select(selector_items, ["en", "hi", "sw"], {"en": 0.1, "hi": 0.7, "sw": 0.4}, 2)

# Synthetic 5-stage matrix validates metric wiring only; it is not experiment data.
dummy_matrix = [
    [0.50, 0.30, 0.25, 0.20, 0.15],
    [0.48, 0.52, 0.32, 0.25, 0.20],
    [0.46, 0.50, 0.54, 0.35, 0.25],
    [0.44, 0.48, 0.52, 0.56, 0.38],
    [0.42, 0.46, 0.50, 0.54, 0.58],
]
dummy_metrics, dummy_table = summarize_performance(dummy_matrix, [0.2] * 5, [item[0] for item in DUMMY_STREAM])
log_record = log_language_accuracy("DRY-RUN", "order-1", 0, "en", dummy_matrix[0][0])

assert len(dummy_batches) == 10
assert len(buffer_sample) == 2
assert len(uniform_sample) == 2 and len(embedding_sample) == 1
assert len(gradient_sample) == 1 and len(prior_sample) == 2
assert abs(dummy_metrics["ACC"] - sum(dummy_matrix[-1]) / 5) < 1e-12
assert log_record in LOCAL_LOG_RECORDS
assert TRAINER_SKELETON.model is None
print({"batches": len(dummy_batches), "sampled_buffer_items": len(buffer_sample), "metrics": dummy_metrics, "logged_record": log_record, "model_weights_loaded": False, "training_started": False})
dummy_table

{'batches': 10, 'sampled_buffer_items': 2, 'metrics': {'ACC': 0.5, 'AA': 0.5, 'BWT': -0.05000000000000002, 'FWT': 0.13749999999999998, 'F': 0.05000000000000002, 'FP': 0.5}, 'logged_record': {'method': 'DRY-RUN', 'order': 'order-1', 'stage': 0, 'language': 'en', 'accuracy': 0.5}, 'model_weights_loaded': False, 'training_started': False}


,language,first_seen_accuracy,final_accuracy
0,en,0.50,0.42
1,hi,0.52,0.46
2,bn,0.54,0.50
3,de,0.56,0.54
4,sw,0.58,0.58
